# CASAS Aruba Smart Home Dataset - Dementia Detection
## Stage 1: Understanding & Loading CASAS Data

**Important:** This project uses the **CASAS Aruba** dataset from https://casas.wsu.edu/datasets/
- Single elderly resident
- Months of longitudinal sensor data
- Labeled activities (Sleep, Meal_Preparation, Eating, etc.)
- Best suited for dementia pattern detection

**Data Format:**
```
2010-11-04 00:03:50.209589  M009  ON   Sleep_Out_of_Bed
2010-11-04 00:03:53.121010  M009  OFF
2010-11-04 00:04:01.209589  M010  ON
```
Each row: `timestamp | sensor_id | state | activity_label (optional)`

In [7]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta
import os

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✓ Libraries imported successfully!")

✓ Libraries imported successfully!


## Stage 1: Load CASAS Aruba Dataset

Loading the pre-processed CSV format directly from Kaggle provides clean, structured data ready for analysis.

In [11]:
import pandas as pd
import numpy as np
import os

DATA_DIR = '/kaggle/input/datasets/ashley6009/casas-smart-home-dataset/data'

df = pd.read_csv(
    os.path.join(DATA_DIR, 'aruba.csv'),
    header=None,
    names=['date', 'time', 'sensor', 'state']
)

# ── Combine date + time into single timestamp ─────────────────────────────
df['timestamp'] = pd.to_datetime(
    df['date'].astype(str) + ' ' + df['time'].astype(str),
    format='mixed'
)
df = df.drop(columns=['date', 'time'])
df = df[['timestamp', 'sensor', 'state']]

# ── Clean whitespace ──────────────────────────────────────────────────────
df['sensor'] = df['sensor'].astype(str).str.strip()
df['state']  = df['state'].astype(str).str.strip()

df = df.sort_values('timestamp').reset_index(drop=True)

print("✓ Loaded successfully!")
print(f"  Rows: {df.shape[0]:,}")
print(f"  Date range: {df['timestamp'].min()} → {df['timestamp'].max()}")
print(f"  Unique sensors: {df['sensor'].nunique()}")
print(f"  Sensor list: {sorted(df['sensor'].unique().tolist())}")
print(f"  States: {df['state'].unique().tolist()}")
print("\nFirst 10 rows:")
print(df.head(10).to_string())

✓ Loaded successfully!
  Rows: 1,602,821
  Date range: 2010-11-04 00:03:50.209589 → 2011-06-11 23:20:35.722380
  Unique sensors: 10
  Sensor list: ['Bathroom', 'Bedroom', 'DiningRoom', 'GuestRoom', 'Kitchen', 'LivingRoom', 'LoungeChair', 'OtherRoom', 'OutsideDoor', 'WorkArea']
  States: ['ON', 'OFF', 'OPEN', 'CLOSE']

First 10 rows:
                   timestamp   sensor state
0 2010-11-04 00:03:50.209589  Bedroom    ON
1 2010-11-04 00:03:57.399391  Bedroom   OFF
2 2010-11-04 02:32:33.351906  Bedroom    ON
3 2010-11-04 02:32:38.895958  Bedroom   OFF
4 2010-11-04 03:42:21.823650  Bedroom    ON
5 2010-11-04 03:42:25.939730  Bedroom   OFF
6 2010-11-04 03:49:52.412755  Bedroom    ON
7 2010-11-04 03:49:57.473649  Bedroom   OFF
8 2010-11-04 04:14:32.835757  Bedroom    ON
9 2010-11-04 04:14:33.203704  Bedroom    ON


## Explore the Data Structure

In [12]:
# Display first 10 rows
print("First 10 rows of data:")
df.head(10)

First 10 rows of data:


,timestamp,sensor,state
0,2010-11-04 00:03:50.209589,Bedroom,ON
1,2010-11-04 00:03:57.399391,Bedroom,OFF
2,2010-11-04 02:32:33.351906,Bedroom,ON
3,2010-11-04 02:32:38.895958,Bedroom,OFF
4,2010-11-04 03:42:21.823650,Bedroom,ON
5,2010-11-04 03:42:25.939730,Bedroom,OFF
6,2010-11-04 03:49:52.412755,Bedroom,ON
7,2010-11-04 03:49:57.473649,Bedroom,OFF
8,2010-11-04 04:14:32.835757,Bedroom,ON
9,2010-11-04 04:14:33.203704,Bedroom,ON


In [13]:
# Check unique activities (these are the labeled behaviors)
activities = df['activity'].dropna().unique()
print(f"Found {len(activities)} unique activities:")
for i, activity in enumerate(activities, 1):
    count = (df['activity'] == activity).sum()
    print(f"  {i:2d}. {activity:<30} ({count:,} occurrences)")

KeyError: 'activity'

In [ ]:
# Check unique sensors
sensors = df['sensor'].unique()
print(f"Found {len(sensors)} unique sensors:")
print(f"Sensors: {sorted(sensors)}")

In [ ]:
# Data info
print("Dataset Information:")
print(f"  Total events: {len(df):,}")
print(f"  Labeled events (with activity): {df['activity'].notna().sum():,}")
print(f"  Unlabeled events: {df['activity'].isna().sum():,}")
print(f"\nSensor states: {df['state'].unique()}")
print(f"\nMissing values:")
print(df.isnull().sum())

In [ ]:
# Time-based analysis
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['date'] = df['timestamp'].dt.date

print("Temporal Distribution:")
print(f"  Events per day (average): {len(df) / df['date'].nunique():.0f}")
print(f"  Most active hour: {df['hour'].mode()[0]}:00")
print(f"  Least active hour: {df.groupby('hour').size().idxmin()}:00")

## Data Visualizations

Now let's visualize sensor activity patterns and temporal distributions

---
## Stage 2: Feature Engineering (Most Important!)

**Goal:** Transform raw sensor events into daily behavioral features that capture routine patterns.

**Key Concept:** Each row = one day in the resident's life. Features capture:
- Activity patterns (kitchen/bedroom/bathroom visits)
- Temporal routines (wake/sleep times, meal windows)
- Dementia-specific indicators (night wandering, meal skipping, sedentary behavior)
- Safety risks (stove usage)

In [ ]:
# Define sensor categories based on CASAS Aruba sensor map
# These map physical sensors to room/function types

KITCHEN_SENSORS  = {'M008', 'M009', 'M010', 'M011', 'M012', 'M013',
                    'M014', 'M015', 'M016', 'M017', 'M018', 'M019',
                    'M020', 'M021', 'M022', 'M023'}
BEDROOM_SENSORS  = {'M001', 'M002', 'M003', 'M004', 'M005'}
BATHROOM_SENSORS = {'M026', 'M027', 'M028', 'M029', 'M030', 'M031'}
DOOR_SENSORS     = {'D001', 'D002', 'D003', 'D004'}

# Stove sensor — critical for gas safety monitoring
STOVE_SENSOR = 'M014'   # adjust to actual stove sensor ID

print("✓ Sensor categories defined")